# 1. Setup

In [ ]:
import sys
from pathlib import Path

In [ ]:
import json
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error as MAE
from xgboost import DMatrix

In [ ]:
PROJECT_ROOT = Path().resolve().parent
RAW_DATA_PATH = "../data/raw_data/"
ADDITIONAL_DATA_PATH = "../data/additional_data/"
PARQUET_DATA_PATH = "../data/parquet_data/"
sys.path.append(str(PROJECT_ROOT))

In [ ]:
from utils.loading import load_parquet_data
from utils.feature_engineering import get_month_splits, drop_split
from utils.data_pipeline import DataPipeline
from utils.model_pipeline import load_or_train_core

In [ ]:
pd.set_option(
    "display.float_format",
    lambda x: f"{x:.2e}" if abs(x) < 0.01 and x != 0 else f"{x:.2f}",
    "display.max_columns",
    100,
    "display.max_rows",
    100,
)
# sns.set_style("whitegrid")

In [ ]:
# SEGMENT_C = ["county", "product_type", "is_business"]
# CATEGORICAL_C = ["county", "product_type", "is_business", "is_consumption"]
# TARGET_C = [
#     "county",
#     "product_type",
#     "is_business",
#     "is_consumption",
#     "datetime",
# ]
# RAND = 10

CATEGORICAL_F = ["county", "product_type", "is_business", "is_consumption"]
RECORD_F = CATEGORICAL_F + ["datetime"]
RAND = 10

In [ ]:
LAG_SPECS = {
    "2d": {
        "3h": ["mean", "count"],
        "6h": ["mean", "count"],
        "24h": ["mean", "median", "std", "count"],
    },
    "3d": {
        "6h": ["mean", "count"],
        "12h": ["mean", "median", "count"],
    },
    "7d": {
        "12h": ["mean", "count"],
        "24h": ["mean", "median", "std", "count"],
    },
}

In [ ]:
dp = DataPipeline(PARQUET_DATA_PATH)
dp.load()
dp.prepare()
dp.merge()
dp.add_features(LAG_SPECS, CATEGORICAL_F, "datetime", "target")

df = dp.df

In [ ]:
# df.info()

# 2. XGB Baseline Model

In [ ]:
# FEATURES_TO_DROP = ["datetime", "data_block_id", "date"]
FEATURES_TO_DROP = ["datetime"]

In [ ]:
BASELINE_MODELS_DIR = Path("../models/xgb_baseline")
BASELINE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
# FH = 7  # weekly retraining
ITERS = 1000
VERBOSE = 100
ESR = 50
baseline_params = {
    "learning_rate": 0.1,
    "max_depth": 7,
    "random_state": RAND,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:absoluteerror",
    "eval_metric": "mae",
    "tree_method": "hist",  # GPU, fastest XGBoost tree-building method
    "device": "cuda",  # GPU
    "nthread": -1,
}

In [ ]:
# def load_train_save(
#     df: pd.DataFrame,
#     split: dict,
#     kind: str,
#     expanding: bool,
#     params: dict,
#     to_drop: list,
#     models_dir: Path,
#     i: int,
#     save: bool = True,
#     num_boost_round: int = 1000,
#     early_stopping_rounds: int = 50,
#     verbose_eval: int = 0,
# ):
#     exp_prefix = "fix" if not expanding else "exp"
#     model_path = models_dir / f"{kind}_{exp_prefix}_{i}.ubj"
#     meta_path = models_dir / f"{kind}_{exp_prefix}_{i}_meta.json"

#     need_to_train = True
#     if model_path.exists() and meta_path.exists():
#         try:
#             with open(meta_path, "r", encoding="utf-8") as fin:
#                 meta = json.load(fin)
#             if (meta.get("train_start") == str(split["train"][0])) and (
#                 meta.get("train_end") == str(split["train"][1])
#             ):
#                 need_to_train = False
#                 booster = xgb.Booster()
#                 booster.load_model(str(model_path))
#             else:
#                 need_to_train = True
#         except Exception:
#             need_to_train = True

#     X_test, y_test = drop_split(df, split["test"], to_drop)
#     dtest = DMatrix(X_test, y_test, enable_categorical=True)
#     del X_test

#     if need_to_train:
#         X_train, y_train = drop_split(df, split["train"], to_drop)
#         dtrain = DMatrix(X_train, y_train, enable_categorical=True)
#         del X_train, y_train

#         evals = [(dtrain, "train")]
#         if "val" in kind:
#             evals.append((dtest, "val"))

#         booster = xgb.train(
#             params=params,
#             dtrain=dtrain,
#             num_boost_round=num_boost_round,
#             evals=evals,
#             early_stopping_rounds=early_stopping_rounds,
#             verbose_eval=verbose_eval,
#         )

#         if save:
#             booster.save_model(str(model_path))
#             meta = {
#                 "train_start": str(split["train"][0]),
#                 "train_end": str(split["train"][1]),
#                 "kind": kind,
#                 "expanding": str(expanding),
#             }
#             with open(meta_path, "w", encoding="utf-8") as fout:
#                 json.dump(meta, fout, ensure_ascii=False, indent=2)

#     return booster, dtest, y_test

In [ ]:
start_ts = df["datetime"].min()
splits = get_month_splits(start_ts, 19, 1, 1, 2)
# april_test = get_month_splits(start_ts, 19, 1, 1, 1)
# may_test = get_month_splits(start_ts, 20, 1, 1, 1)
print(*splits, sep="\n")

{'train': (Timestamp('2021-09-01 00:00:00'), Timestamp('2023-03-31 23:00:00')), 'test': (Timestamp('2023-04-01 00:00:00'), Timestamp('2023-04-30 23:00:00'))}
{'train': (Timestamp('2021-09-01 00:00:00'), Timestamp('2023-04-30 23:00:00')), 'test': (Timestamp('2023-05-01 00:00:00'), Timestamp('2023-05-31 23:00:00'))}


In [ ]:
baseline_scores = []
baseline_results = np.zeros(len(df), dtype=np.float32)
baseline_mae_history = []

for i, split in enumerate(splits):
    # 1. Prepare ONLY the test data (for prediction/scoring)
    test_mask = (df["datetime"] >= split["test"][0]) & (df["datetime"] <= split["test"][1])
    
    # Ensure feature consistency: drop datetime and target
    feature_cols = [c for c in df.columns if c not in FEATURES_TO_DROP + ["target"]]
    X_test = df.loc[test_mask, feature_cols]
    y_test = df.loc[test_mask, "target"]
    
    # We still need a DMatrix for the test set to use booster.predict()
    dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)

    # 2. Call the "Lazy" Pipeline
    # This handles temporal splitting, eval-set sampling, and DMatrix creation internally!
    booster, was_trained, history = load_or_train_core(
        models_dir="../models",
        notebook="hyperparameter_optimization",
        purpose="baseline",
        split=split,
        model_params=baseline_params,
        df=df,
        target_col="target",
        drop_cols=FEATURES_TO_DROP,
        eval_sample_size=100_000, # Same as your sklearn baseline
        num_boost_round=ITERS,
        early_stopping_rounds=ESR,
        verbose_eval=VERBOSE,
        random_state=RAND
    )

    # 3. Store history and predictions
    baseline_mae_history.append(history)
    preds = booster.predict(dtest)

    # Store results and score
    baseline_results[X_test.index] = preds
    score = MAE(y_test, preds)
    baseline_scores.append({"Split": i, "MAE": score})

    status = "Retrained" if was_trained else "Loaded from Cache"
    print(f"Split {i} ({status}) MAE: {score:.4f}")

print(f"\nOverall Baseline MAE: {np.mean([s['MAE'] for s in baseline_scores]):.4f}")

[0]	train-mae:252.64566	eval-mae:255.15498
[100]	train-mae:45.23488	eval-mae:45.16291
[200]	train-mae:40.93193	eval-mae:41.10687
[300]	train-mae:38.09898	eval-mae:38.54232
[400]	train-mae:36.13567	eval-mae:36.75435
[500]	train-mae:34.84191	eval-mae:35.63282
[600]	train-mae:33.51636	eval-mae:34.48714
[700]	train-mae:32.62674	eval-mae:33.72728
[800]	train-mae:31.97416	eval-mae:33.17301
[900]	train-mae:31.31346	eval-mae:32.61371
[999]	train-mae:30.77706	eval-mae:32.16624
Split 0 (Retrained) MAE: 70.3225
[0]	train-mae:255.44398	eval-mae:255.30636
[100]	train-mae:46.76873	eval-mae:47.31535
[200]	train-mae:42.52334	eval-mae:43.19866
[300]	train-mae:39.18819	eval-mae:40.03293
[400]	train-mae:37.15608	eval-mae:38.14020
[500]	train-mae:35.67435	eval-mae:36.79476
[600]	train-mae:34.44897	eval-mae:35.71307
[700]	train-mae:33.54436	eval-mae:34.88195
[800]	train-mae:32.87112	eval-mae:34.29090
[900]	train-mae:32.37621	eval-mae:33.88578
[999]	train-mae:31.87431	eval-mae:33.47133
Split 1 (Retrained) M

In [ ]:
baseline_scores
# [{'Split': 0, 'MAE': 36.081600189208984},
#  {'Split': 1, 'MAE': 36.5767707824707},
#  {'Split': 2, 'MAE': 59.67155456542969}]

[{'Split': 0, 'MAE': 36.081600189208984},
 {'Split': 1, 'MAE': 36.5767707824707},
 {'Split': 2, 'MAE': 59.67155456542969}]

In [ ]:
qwe

In [ ]:
# exp_splits = [
#     {
#         "train": get_month_splits(start_ts, 6, 1, 1, 1)[0]["train"],
#         "test": (
#             pd.Timestamp("2023-03-01 00:00:00"),
#             pd.Timestamp("2023-03-31 23:00:00"),
#         ),
#     }
# ]

# exp_splits

[{'train': (Timestamp('2021-09-01 00:00:00'),
   Timestamp('2022-02-28 23:00:00')),
  'test': (Timestamp('2023-03-01 00:00:00'),
   Timestamp('2023-03-31 23:00:00'))}]

In [ ]:
# exp_scores = []
# exp_results = np.zeros(len(df), dtype=np.float32)
# exp_mae_history = []

# for i, split in enumerate(exp_splits):
#     # 1. Prepare ONLY the test data (for prediction/scoring)
#     test_mask = (df["datetime"] >= split["test"][0]) & (
#         df["datetime"] <= split["test"][1]
#     )

#     # Ensure feature consistency: drop datetime and target
#     feature_cols = [
#         c for c in df.columns if c not in FEATURES_TO_DROP + ["target"]
#     ]
#     X_test = df.loc[test_mask, feature_cols]
#     y_test = df.loc[test_mask, "target"]

#     # We still need a DMatrix for the test set to use booster.predict()
#     dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)

#     # 2. Call the "Lazy" Pipeline
#     # This handles temporal splitting, eval-set sampling, and DMatrix creation internally!
#     booster, was_trained, history = load_or_train_core(
#         models_dir="../models",
#         notebook="hyperparameter_optimization",
#         purpose="small_train",
#         split=split,
#         model_params=baseline_params,
#         df=df,
#         target_col="target",
#         drop_cols=FEATURES_TO_DROP,
#         eval_sample_size=0,  # Same as your sklearn baseline
#         num_boost_round=ITERS,
#         early_stopping_rounds=ESR,
#         verbose_eval=VERBOSE,
#         random_state=RAND,
#     )

#     # 3. Store history and predictions
#     exp_mae_history.append(history)
#     preds = booster.predict(dtest)

#     # Store results and score
#     exp_results[X_test.index] = preds
#     score = MAE(y_test, preds)
#     exp_scores.append({"Split": i, "MAE": score})

#     status = "Retrained" if was_trained else "Loaded from Cache"
#     print(f"Split {i} ({status}) MAE: {score:.4f}")

# print(f"\nOverall Baseline MAE: {np.mean([s['MAE'] for s in exp_scores]):.4f}")

[0]	train-mae:232.94452
[100]	train-mae:31.38385
[200]	train-mae:28.24570
[300]	train-mae:26.04266
[400]	train-mae:24.42778
[500]	train-mae:23.10824
[600]	train-mae:22.13877
[700]	train-mae:21.27145
[800]	train-mae:20.55057
[900]	train-mae:19.98378
[999]	train-mae:19.56907
Split 0 (Retrained) MAE: 74.0947

Overall Baseline MAE: 74.0947


In [ ]:
# booster, was_trained, history = load_or_train_core(
#     models_dir="../models",
#     notebook="optuna",
#     purpose="baseline",
#     model_params=baseline_params,
#     split=split_bounds,
#     dtrain=dtrain,           # Your DMatrix
#     evals=[(dval, "val")],   # Your evaluation list
#     num_boost_round=ITERS,
#     early_stopping_rounds=ESR,
#     verbose_eval=100
# )


# preds = booster.predict(dtest)

In [ ]:
# scores = []
# results = np.zeros(len(df), dtype=np.float32)
# xgb_mae_history = []

# for i, split in enumerate(splits):

#     train_mask = (df["datetime"] >= split["train"][0]) & (
#         df["datetime"] <= split["train"][1]
#     )
#     test_mask = (df["datetime"] >= split["test"][0]) & (
#         df["datetime"] <= split["test"][1]
#     )

#     X_train = df.loc[
#         train_mask, df.columns.difference(FEATURES_TO_DROP, sort=False)
#     ]
#     y_train = df.loc[train_mask, "target"]

#     # X_test / y_test (this month's test data)
#     X_test = df.loc[
#         test_mask, df.columns.difference(FEATURES_TO_DROP, sort=False)
#     ]
#     y_test = df.loc[test_mask, "target"]

#     # 2. Create DMatrix objects (Core API)
#     dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
#     dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)

#     # Use dtest as validation for the baseline (or a separate sample if you prefer)
#     evals = [(dtrain, "train"), (dtest, "val")]
#     # 3. Call the pipeline
#     booster, was_trained, history = load_or_train_core(
#         models_dir="../models",
#         notebook="optuna",
#         purpose="baseline",
#         model_params=baseline_params,
#         split=split,  # <--- Pass the current split dict
#         dtrain=dtrain,
#         evals=evals,
#         num_boost_round=ITERS,
#         early_stopping_rounds=ESR,
#         verbose_eval=100,
#     )
#     # 4. Store history and predictions
#     xgb_mae_history.append(history)
#     preds = booster.predict(dtest)

#     # Store predictions in the global results array
#     results[X_test.index] = preds

#     # Calculate and store score
#     score = MAE(y_test, preds)
#     scores.append({"Split": i, "MAE": score})

#     print(f"Split {i} MAE: {score:.4f}")
# # Final result summary
# print(f"Overall Baseline MAE: {np.mean([s['MAE'] for s in scores]):.4f}")

In [ ]:
# for split in enumerate(splits):
#     # train_mask = (df["datetime"] >= split["train"][0]) & (
#     #     df["datetime"] <= split["train"][1]
#     # )
#     X_train, y_train = (
#         df.loc[
#             train_mask,
#             df.columns.difference(FEATURES_TO_DROP, sort=False),
#         ],
#         df.loc[train_mask, "target"],
#     )
#     test_mask = (df["datetime"] >= split["test"][0]) & (
#         df["datetime"] <= split["test"][1]
#     )
#     X_test, y_test = (
#         df.loc[
#             test_mask,
#             df.columns.difference(FEATURES_TO_DROP, sort=False),
#         ],
#         df.loc[test_mask, "target"],
#     )
#     # Eval set for tracking progress over iterations
#     eval_idx = X_train.sample(100_000, random_state=RAND).index
#     eval_set = [(X_train.loc[eval_idx], y_train.loc[eval_idx])]

#     for model_name, model_cls, model_params in [
#         ("xgb", XGBRegressor, xgb_p),
#         ("lgbm", LGBMRegressor, lgbm_p),
#         ("cb", CatBoostRegressor, cb_p),
#     ]:

#         model, was_trained, history = load_or_train_sklearn(
#             models_dir="../models",
#             notebook="baseline_comparison",
#             purpose="small_baseline",
#             model_name=model_name,
#             model_cls=model_cls,
#             model_params=model_params,
#             split=split,
#             X_train=X_train,
#             y_train=y_train,
#             eval_set=eval_set,
#             cat_cols=cat_cols,
#         )
#         # append history
#         if model_name == "xgb":
#             xgb_mae_history.append(history)
#         elif model_name == "lgbm":
#             lgbm_mae_history.append(history)
#         elif model_name == "cb":
#             cb_mae_history.append(history)

#         preds = model.predict(X_test)
#         results[model_name][X_test.index] = preds
#         scores.append(
#             {
#                 "Split": i,
#                 "Model": model_name,
#                 "MAE": MAE(y_test, preds),
#             }
#         )
#     # naive baseline
#     scores.append(
#         {
#             "Split": i,
#             "Model": "naive",
#             "MAE": MAE(
#                 y_test.loc[X_test["target_lag_2d"].notna()],
#                 X_test["target_lag_2d"].dropna(),
#             ),
#         }
#     )

In [ ]:
# mae_baseline_test = []

In [ ]:
# for i, split in enumerate(april_test):
#     booster, dtest, y_test = load_train_save(
#         df,
#         split,
#         "test_2m",
#         True,
#         baseline_params,
#         FEATURES_TO_DROP,
#         BASELINE_MODELS_DIR,
#         i,
#         True,
#         ITERS,
#         ESR,
#         VERBOSE,
#     )
#     preds = booster.predict(dtest)
#     mae_baseline_test.append(MAE(y_test, preds))

In [ ]:
# mae_baseline_test

# 3. Optuna Search

In [ ]:
start_ts = df["datetime"].min()

In [ ]:
# Splits for 3 models with different time period
optuna_train_lv = get_month_splits(start_ts, 12, 1, 3, 3)
optuna_train_lv

In [ ]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lambda": trial.suggest_float("lambda", 0.001, 100),
        "alpha": trial.suggest_float("alpha", 0.001, 100),
        "gamma": trial.suggest_float("gamma", 0, 10),
        "grow_policy": trial.suggest_categorical(
            "grow_policy", ["depthwise", "lossguide"]
        ),
        "tree_method": "hist",
        "device": "cuda",
        "objective": "reg:absoluteerror",
        "eval_metric": "mae",
    }
    # num_boost_round = trial.suggest_int("num_boost_round", 500, 2500, step=500)
    num_boost_round = 500

    cv_scores = np.empty(len(optuna_train_lv))

    for i, split in enumerate(optuna_train_lv):
        params["random_state"] = RAND + i
        X_train, y_train = drop_split(df, split["train"], FEATURES_TO_DROP)
        dtrain = DMatrix(X_train, y_train, enable_categorical=True)
        del X_train, y_train

        X_val, y_val = drop_split(df, split["test"], FEATURES_TO_DROP)
        dval = DMatrix(X_val, y_val, enable_categorical=True)
        del X_val

        evals = [(dtrain, "train"), (dval, "val")]

        booster = xgb.train(
            params=params,
            dtrain=dtrain,
            evals=evals,
            num_boost_round=num_boost_round,
            early_stopping_rounds=ESR,
            verbose_eval=VERBOSE,
        )

        preds = booster.predict(dval)
        cv_scores[i] = MAE(y_val, preds)

    return np.mean(cv_scores)

In [ ]:
# STORAGE = "sqlite:///../optuna_db/optuna_study_long_val_rand_incr.db"
# n_trials = 120
# n_startup_trials = 20

STORAGE = "sqlite:///../optuna_db/optuna_study_long_val_rand_incr_dp.db"
n_trials = 8  # 120
n_startup_trials = 20

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)

In [ ]:
study_lvri = optuna.create_study(
    storage=STORAGE,
    sampler=TPESampler(n_startup_trials=n_startup_trials, multivariate=True),
    pruner=optuna.pruners.SuccessiveHalvingPruner(),
    study_name="xgb_optuna",
    direction="minimize",
    load_if_exists=True,
)
existing_trials = len(study_lvri.trials)

if existing_trials >= n_trials:
    print("Number of existing trials >= n_trials. Skipping optimization.")
else:
    remaining = n_trials - existing_trials
    print(f"Run {remaining} trials to reach {n_trials}")
    study_lvri.optimize(
        objective,
        n_trials=remaining,
        show_progress_bar=True,
        n_jobs=1,
    )

In [ ]:
study_lvri.best_value

In [ ]:
best_params = study_lvri.best_params.copy()
# num_boost_round = best_params.pop("num_boost_round")
num_boost_round = 500
best_params.update({"random_state": RAND})
for k, v in best_params.items():
    print(k, v)
print(num_boost_round)

In [ ]:
april_test

In [ ]:
mae_tests = []
OPTUNA_MODELS_DIR = Path("../models/xgb_optuna")
OPTUNA_MODELS_DIR.mkdir(parents=True, exist_ok=True)

for i, split in enumerate(april_test):
    X_train, y_train = drop_split(df, split["train"], FEATURES_TO_DROP)
    dtrain = xgb.DMatrix(X_train, y_train, enable_categorical=True)
    del X_train, y_train
    
    booster = xgb.train(
        params=best_params,
        dtrain=dtrain,
        num_boost_round=num_boost_round,
        verbose_eval=True,
    )
    del dtrain

    model_path = OPTUNA_MODELS_DIR / f"optuna_split_{i}.ubj"
    meta_path = OPTUNA_MODELS_DIR / f"optuna_split_{i}_meta.json"

    meta = {
        "train_start": str(split["train"][0]),
        "train_end": str(split["train"][1]),
        "test_start": str(split["test"][0]),
        "test_end": str(split["test"][1]),
        "params": best_params,
        "num_boost_round": num_boost_round,
    }

    booster.save_model(model_path)
    with open(meta_path, "w", encoding="utf-8") as fout:
        json.dump(meta, fout, ensure_ascii=False, indent=2)

    X_test, y_test = drop_split(df, split["test"], FEATURES_TO_DROP)
    dtest = xgb.DMatrix(X_test, enable_categorical=True)
    del X_test
    preds = booster.predict(dtest)
    mae_tests.append(MAE(y_test, preds))

In [ ]:
print(mae_tests)